# 🚀 Colab Turbo Downloader (No-Code UI)

### How to use:
1. Fill the settings below
2. Click **Runtime → Run all**
3. Approve Google Drive mount
4. Wait for **Download complete**

Only the settings cell is editable. All other cells are run-only.

In [ ]:

#@title 🔧 Downloader Settings

DOWNLOAD_URL = ""  #@param {type:"string"}
OUTPUT_FILENAME = ""  #@param {type:"string"}
PARALLEL_CONNECTIONS = 8  #@param {type:"integer"}
SAVE_FOLDER_IN_DRIVE = "/content/drive/MyDrive/Downloads"  #@param {type:"string"}

#@markdown ### ⚙️ Advanced Settings (optional)
SHOW_ADVANCED_SETTINGS = False  #@param {type:"boolean"}
RETRY_ATTEMPTS = 4  #@param {type:"integer"}
RETRY_BACKOFF_SECONDS = 1.5  #@param {type:"number"}
REQUEST_TIMEOUT_SECONDS = 30  #@param {type:"integer"}
CHUNK_SIZE_KB = 1024  #@param {type:"integer"}

print("✅ Settings ready. Now run all cells.")


In [ ]:

#@title 🔗 Mount & Setup

import math, os, time, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

def human_bytes(x):
    for unit in ['B','KB','MB','GB','TB']:
        if x < 1024:
            return f"{x:.2f} {unit}"
        x /= 1024

def ensure_dir(p):
    Path(p).mkdir(parents=True, exist_ok=True)

ensure_dir(SAVE_FOLDER_IN_DRIVE)

print("✅ Environment ready")


In [ ]:

#@title 🚀 Start Download

import threading

def get_info(url):
    r = requests.head(url, allow_redirects=True)
    size = r.headers.get("Content-Length")
    return int(size) if size and size.isdigit() else None, "bytes" in r.headers.get("Accept-Ranges","").lower()

def download():
    url = DOWNLOAD_URL.strip()
    if not url:
        raise ValueError("Please enter a download URL.")

    name = OUTPUT_FILENAME.strip() or url.split("/")[-1] or "file.bin"
    path = os.path.join(SAVE_FOLDER_IN_DRIVE, name)

    total, ranges = get_info(url)
    print("📦 Size:", human_bytes(total) if total else "Unknown")

    chunk = CHUNK_SIZE_KB * 1024
    retries = RETRY_ATTEMPTS if SHOW_ADVANCED_SETTINGS else 3

    pbar = tqdm(total=total, unit='B', unit_scale=True)

    def dl_part(start, end, file):
        headers = {"Range": f"bytes={start}-{end}"} if end else {}
        for _ in range(retries):
            try:
                with requests.get(url, headers=headers, stream=True) as r:
                    with open(file, "ab") as f:
                        for c in r.iter_content(chunk):
                            if c:
                                f.write(c)
                                pbar.update(len(c))
                return
            except:
                time.sleep(1)

    if total and ranges and PARALLEL_CONNECTIONS > 1:
        part_size = math.ceil(total / PARALLEL_CONNECTIONS)
        tmp = path + ".parts"
        os.makedirs(tmp, exist_ok=True)

        with ThreadPoolExecutor(max_workers=PARALLEL_CONNECTIONS) as ex:
            futures = []
            for i in range(PARALLEL_CONNECTIONS):
                s = i * part_size
                e = min(total-1, s + part_size - 1)
                futures.append(ex.submit(dl_part, s, e, f"{tmp}/p{i}"))
            for f in as_completed(futures):
                f.result()

        with open(path, "wb") as out:
            for i in range(PARALLEL_CONNECTIONS):
                part = f"{tmp}/p{i}"
                if os.path.exists(part):
                    out.write(open(part, "rb").read())
                    os.remove(part)
        os.rmdir(tmp)
    else:
        dl_part(0, None, path)

    pbar.close()
    print("✅ Download complete")
    print("📁 Saved to:", path)

download()
